# Lab 8: Anomaly detection

<a target="_blank" href="https://colab.research.google.com/github/drchadvidden/courseMaterials/blob/main/UnsupervisedLearning/Labs/Lab%208/Lab_8.ipynb">
<img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# Lab Instructions

Run each of the coding cells. For tutorial example cells, understand the commands and check that the outputs make sense. For exercise cells, write your own code where indicated to generate the correct output. Give text explanations where indicated.

### Submission:
Complete the following notebook in order. Once done, save the notebook, print the file as a .pdf, and upload the resulting file to the Canvas course assignment.

### Rubric:
15 total points, 5 points to running tutorial example cells and saving outputs, 10 points for completing exercises.

### Deadline:
Tuesday at midnight after the lab is assigned.

# Tutorial: Statistical Anomaly Detection Explained

The statistical approach assumes that **normal data points follow a known probability distribution**, typically a **multivariate Gaussian (multinormal) distribution**. Anomalies are points that are **unlikely under this model**.

### 1. Gaussian Parameters
Given a dataset $X = \{x_1, x_2, \dots, x_n\}$ with $d$ features:

- **Mean vector:**  
$$
\mu = \frac{1}{n}\sum_{i=1}^n x_i
$$

- **Covariance matrix:**  
$$
\Sigma = \frac{1}{n-1}\sum_{i=1}^n (x_i - \mu)(x_i - \mu)^T
$$

### 2. Multivariate Gaussian Probability Density
For any point $x \in \mathbb{R}^d$:

$$
p(x) = \frac{1}{(2\pi)^{d/2} |\Sigma|^{1/2}} \exp\Bigg( -\frac{1}{2} (x - \mu)^T \Sigma^{-1} (x - \mu) \Bigg)
$$

Points with **very low $p(x)$** are likely anomalies.

### 3. Mahalanobis Distance
Equivalent distance-based anomaly score:

$$
D_M(x) = \sqrt{(x - \mu)^T \Sigma^{-1} (x - \mu)}
$$

- **Squared Mahalanobis distance** is proportional to the exponent in the Gaussian pdf.

### 4. Anomaly Detection
- Flag a point as anomalous if  
$$
D_M(x)^2 > \chi^2_{d, 0.99}
$$  
using the Chi-square distribution with $d$ degrees of freedom (2 for our 2D example).

**Advantages:** simple, interpretable, directly linked to probability.  
**Limitations:** sensitive to non-Gaussian distributions; covariance estimation unstable in high dimensions with few points.

## Iris Dataset Overview

The **Iris dataset** is a classic dataset in statistics and machine learning. It contains **150 samples of iris flowers** from three species (`setosa`, `versicolor`, `virginica`). Each sample has **4 numeric features**:

1. Sepal length (cm)  
2. Sepal width (cm)  
3. Petal length (cm)  
4. Petal width (cm)  

For this lab, we will use **only the first two features** (sepal length and sepal width) to keep the data **2D** and easy to visualize.  

The goal is to detect **points that are unusual or far from the main cluster** using a **statistical anomaly detection approach** (Mahalanobis distance).

In [ ]:
# ----------------------------
# Statistical Anomaly Detection on Iris (2D)
# ----------------------------

import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris
from scipy.stats import chi2
from numpy.linalg import inv

# ----------------------------
# Load Iris dataset (2D)
# ----------------------------
iris = load_iris()
X = iris.data[:, :2]  # first two features
feature_names = iris.feature_names[:2]

# ----------------------------
# Step 1: Compute mean and covariance
# ----------------------------
mu = X.mean(axis=0)
Sigma = np.cov(X, rowvar=False)
Sigma_inv = inv(Sigma)

# ----------------------------
# Step 2: Compute Mahalanobis distances
# ----------------------------
def mahalanobis(x, mu, Sigma_inv):
    diff = x - mu
    return np.dot(np.dot(diff, Sigma_inv), diff.T)

distances = np.array([mahalanobis(x, mu, Sigma_inv) for x in X])

# ----------------------------
# Step 3: Threshold (Chi-square, 99% confidence)
# ----------------------------
threshold = chi2.ppf(0.99, df=2)
is_anomaly = distances > threshold

# ----------------------------
# Step 4: Visualization
# ----------------------------
plt.figure(figsize=(6,6))
plt.scatter(X[~is_anomaly, 0], X[~is_anomaly, 1], s=40, label="Normal")
plt.scatter(X[is_anomaly, 0], X[is_anomaly, 1], color="red", s=60, label="Anomaly")
plt.xlabel(feature_names[0])
plt.ylabel(feature_names[1])
plt.title("Mahalanobis Anomaly Detection on Iris (2D)")
plt.legend()
plt.show()

print(f"Chi-square threshold (0.99): {threshold:.2f}")
print("Number of anomalies detected:", is_anomaly.sum())

## Visualizing Anomalies, Probabilities, and Gaussian Contours

After computing Mahalanobis distances and detecting anomalies, it is helpful to **visualize the results**:

1. **Scatter plot with Gaussian contours:** Shows the 2D Gaussian fit over the data.  
   - Points are colored by their **probability density** $p(x)$.  
   - **Red points** are anomalies detected using the Chi-square threshold.  

2. **Mahalanobis distance bar chart:** Visualizes the distance of each sample from the mean, with the **threshold line** indicating anomalies.  

These visualizations make it easier to **interpret which points are anomalous** and how they relate to the estimated Gaussian distribution.

In [ ]:
# ----------------------------
# Additional Visualizations: Mahalanobis distances, Gaussian contour
# ----------------------------

from scipy.stats import multivariate_normal

# 1. Compute probability density p(x) for each point
rv = multivariate_normal(mean=mu, cov=Sigma)
px = rv.pdf(X)

# 2. Create a grid for contour plotting
x = np.linspace(X[:,0].min() - 1, X[:,0].max() + 1, 100)
y = np.linspace(X[:,1].min() - 1, X[:,1].max() + 1, 100)
X_grid, Y_grid = np.meshgrid(x, y)
pos = np.dstack((X_grid, Y_grid))
Z = rv.pdf(pos)

# ----------------------------
# Plot data points, anomalies, and Gaussian contour
# ----------------------------
plt.figure(figsize=(7,7))
plt.contour(X_grid, Y_grid, Z, levels=10, cmap='Blues', alpha=0.6)  # Gaussian contour
scatter = plt.scatter(X[:,0], X[:,1], c=px, cmap='viridis', s=50, label='Points')
plt.scatter(X[is_anomaly,0], X[is_anomaly,1], color='red', s=80, label='Anomalies', edgecolor='k')
plt.xlabel(feature_names[0])
plt.ylabel(feature_names[1])
plt.title("Mahalanobis & Gaussian Density Visualization")
plt.colorbar(scatter, label='p(x)')
plt.legend()
plt.show()

# ----------------------------
# Optionally, plot Mahalanobis distances as a separate bar chart
# ----------------------------
plt.figure(figsize=(8,4))
plt.bar(range(len(distances)), distances, color='skyblue', edgecolor='k')
plt.axhline(threshold, color='red', linestyle='--', label='Chi-square threshold')
plt.xlabel("Sample index")
plt.ylabel("Mahalanobis distance")
plt.title("Mahalanobis Distances for Iris Samples")
plt.legend()
plt.show()

# Tutorial: Distance- and Density-Based Anomaly Detection

In this section, we detect anomalies using **distance-based** and **density-based** approaches. Unlike the statistical method, these approaches do **not assume a specific distribution** for the data.

---

## Distance-Based Approach

The idea is simple:
- A point is anomalous if it is **far from its neighbors**.

For a point $x_i$, let $N_k(x_i)$ denote its set of $k$ nearest neighbors.  
We define the **distance-based anomaly score** as the average distance to its neighbors:

$$
\text{Dist}(x_i) = \frac{1}{k} \sum_{x_j \in N_k(x_i)} \|x_i - x_j\|
$$

- Points with **large $\text{Dist}(x_i)$** are considered anomalies.

---

## Density-Based Approach

Instead of distance, we measure how **dense the local neighborhood** is.

First define a **local density estimate**:

$$
\text{Density}(x_i) = \frac{1}{\text{Dist}(x_i)}
$$

Then compare this to the density of neighboring points. The **relative density score** is:

$$
\text{RelDensity}(x_i) = \frac{\frac{1}{k} \sum_{x_j \in N_k(x_i)} \text{Density}(x_j)}{\text{Density}(x_i)}
$$

- If $\text{RelDensity}(x_i) \approx 1$: point has similar density to neighbors  
- If $\text{RelDensity}(x_i) \gg 1$: point is **less dense than neighbors** → anomaly  

---

### Key Idea

- **Distance-based:** “Is this point far away?”  
- **Density-based:** “Is this point in a sparse region compared to nearby points?”  

These methods may **agree or disagree**, depending on the structure of the data.

---

### Implementation Notes

- We use **Euclidean distance**:
$$
\|x_i - x_j\| = \sqrt{\sum_{m=1}^d (x_{im} - x_{jm})^2}
$$

- The parameter $k$ controls the **local neighborhood size** and can affect which points are detected as anomalies.

---

We now apply both methods to the **Iris dataset (2D)** and compare the detected anomalies.

In [ ]:
# ----------------------------
# Distance- and Density-Based Anomaly Detection on Iris (2D)
# ----------------------------
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris
from sklearn.neighbors import NearestNeighbors

# ----------------------------
# 1) Load Iris dataset (2D)
# ----------------------------
iris = load_iris()
X = iris.data[:, :2]  # first two features
feature_names = iris.feature_names[:2]

# ----------------------------
# 2) Compute distance-based anomaly scores (k-NN)
# ----------------------------
k = 5 # feel free to try various values of k
nbrs = NearestNeighbors(n_neighbors=k+1).fit(X)
distances, indices = nbrs.kneighbors(X)

# exclude self-distance (first column)
k_distances = distances[:, 1:]
distance_score = k_distances.mean(axis=1)

# ----------------------------
# 3) Improved density-based score (relative density)
# ----------------------------

# local density = inverse of average distance
density = 1 / distance_score

# average density of neighbors
neighbor_density = np.array([
    density[indices[i, 1:]].mean() for i in range(len(X))
])

# relative density score
rel_density_score = neighbor_density / density

# ----------------------------
# 4) Threshold to flag anomalies
# ----------------------------
num_anomalies = 5

# Distance-based: large distance = anomaly
threshold_distance = np.sort(distance_score)[-num_anomalies]
is_anomaly_distance = distance_score >= threshold_distance

# Density-based: high relative density ratio = anomaly
threshold_density = np.sort(rel_density_score)[-num_anomalies]
is_anomaly_density = rel_density_score >= threshold_density

# ----------------------------
# 5) Visualization (scatter)
# ----------------------------
plt.figure(figsize=(7,7))
plt.scatter(X[:,0], X[:,1], s=50, label='Points')

# Distance anomalies (red)
plt.scatter(X[is_anomaly_distance,0], X[is_anomaly_distance,1],
            color='red', s=80, label='Distance Anomaly', edgecolor='k')

# Density anomalies (green circles)
plt.scatter(X[is_anomaly_density,0], X[is_anomaly_density,1],
            facecolors='none', edgecolors='green', s=120, label='Density Anomaly')

plt.xlabel(feature_names[0])
plt.ylabel(feature_names[1])
plt.title("Distance vs Density Anomaly Detection (Iris 2D)")
plt.legend()
plt.show()

# ----------------------------
# 6) Distance score plot
# ----------------------------
plt.figure(figsize=(8,4))
plt.bar(range(len(distance_score)), distance_score, edgecolor='k')
plt.axhline(threshold_distance, linestyle='--', label='Distance Threshold')
plt.xlabel("Sample index")
plt.ylabel("Average k-NN distance")
plt.title("Distance-Based Anomaly Scores")
plt.legend()
plt.show()

# ----------------------------
# 7) Relative density score plot
# ----------------------------
plt.figure(figsize=(8,4))
plt.bar(range(len(rel_density_score)), rel_density_score, edgecolor='k')
plt.axhline(threshold_density, linestyle='--', label='Density Threshold')
plt.xlabel("Sample index")
plt.ylabel("Relative Density Score")
plt.title("Density-Based Anomaly Scores")
plt.legend()
plt.show()

## Choosing a Threshold via Elbow Plots

Another way to choose a threshold is to examine the **sorted anomaly scores**. By plotting the scores in increasing order, we can look for a **sharp change (an “elbow”)** in the curve.

- Points beyond the elbow often correspond to **potential anomalies**.  
- This provides a **data-driven way** to select a threshold, rather than fixing the number of anomalies in advance.  

In the plots below:
- The **vertical line** shows the cutoff based on selecting the top $k$ anomalies.  
- The **horizontal line** shows the corresponding **score threshold**.  

Comparing these lines with the shape of the curve helps assess whether the chosen threshold is reasonable.

In [ ]:
# ----------------------------
# 8) Elbow plots with threshold lines
# ----------------------------

# sort scores
sorted_dist = np.sort(distance_score)
sorted_density = np.sort(rel_density_score)

# index where anomalies start
cutoff_index = len(sorted_dist) - num_anomalies

# ----------------------------
# Distance elbow plot
# ----------------------------
plt.figure(figsize=(6,4))
plt.plot(sorted_dist, marker='o')

# vertical line (top-k cutoff)
plt.axvline(cutoff_index, linestyle='--', label='Top-k cutoff')

# horizontal line (threshold value)
plt.axhline(threshold_distance, linestyle='--', label='Threshold value')

plt.xlabel("Sorted index")
plt.ylabel("Distance score")
plt.title("Elbow Plot: Distance-Based Scores")
plt.legend()
plt.show()

# ----------------------------
# Density elbow plot
# ----------------------------
plt.figure(figsize=(6,4))
plt.plot(sorted_density, marker='o')

# vertical line (top-k cutoff)
plt.axvline(cutoff_index, linestyle='--', label='Top-k cutoff')

# horizontal line (threshold value)
plt.axhline(threshold_density, linestyle='--', label='Threshold value')

plt.xlabel("Sorted index")
plt.ylabel("Relative density score")
plt.title("Elbow Plot: Density-Based Scores")
plt.legend()
plt.show()

# Tutorial: Clustering-Based Anomaly Detection

Another approach to finding anomalies is to use **clustering methods**. The idea is:

- Points **far from cluster centers** (K-Means) or  
- Points labeled as **noise by a density-based clustering** (DBSCAN)  

are considered anomalous.

## K-Means
- Assigns each point to the nearest cluster center.  
- Compute the distance from each point to its center:  
$$
d_i = \| x_i - c_{cluster(i)} \|
$$  
- Points with the **largest distances** are flagged as anomalies.  
- Caveat: points lying **between clusters** may not be far from any center, so they can be missed.

## DBSCAN
- Groups points into **dense regions** and labels sparse points as noise.  
- Points with label `-1` are considered anomalies.  
- This method can capture points **between clusters** better than K-Means.

### Visualization
- Red points → anomalies based on K-Means distance  
- Green points → anomalies flagged by DBSCAN  
- Orange X → cluster centers  

This gives a **complementary view**: K-Means focuses on distance to center, while DBSCAN focuses on **local density and sparsity**.

In [ ]:
# ----------------------------
# Clustering-Based Anomaly Detection on Iris (2D)
# ----------------------------
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris
from sklearn.cluster import KMeans, DBSCAN
from sklearn.metrics import pairwise_distances

# ----------------------------
# 1) Load Iris dataset (2D)
# ----------------------------
iris = load_iris()
X = iris.data[:, :2]  # first two features
feature_names = iris.feature_names[:2]

# ----------------------------
# 2) K-Means Based Anomaly Detection
# ----------------------------
k_clusters = 3  # Iris has 3 species

kmeans = KMeans(n_clusters=k_clusters, random_state=42)
labels_km = kmeans.fit_predict(X)
centers = kmeans.cluster_centers_

# compute distance of each point to its cluster center
dist_to_center = np.array([np.linalg.norm(X[i] - centers[labels_km[i]]) for i in range(len(X))])

# flag top-n farthest points as anomalies
num_anomalies = 5
threshold_km = np.sort(dist_to_center)[-num_anomalies]
is_anomaly_km = dist_to_center >= threshold_km

# ----------------------------
# 3) DBSCAN-Based Anomaly Detection
# ----------------------------
dbscan = DBSCAN(eps=0.35, min_samples=5)  # eps tuned for small 2D data
labels_db = dbscan.fit_predict(X)

# DBSCAN labels: -1 = noise (anomaly)
is_anomaly_db = labels_db == -1

# ----------------------------
# 4) Visualization
# ----------------------------
plt.figure(figsize=(7,7))

# Plot all points
plt.scatter(X[:,0], X[:,1], c='skyblue', s=50, label='Points')

# K-Means anomalies (red)
plt.scatter(X[is_anomaly_km,0], X[is_anomaly_km,1],
            color='red', s=80, label='K-Means Anomaly', edgecolor='k')

# DBSCAN anomalies (green)
plt.scatter(X[is_anomaly_db,0], X[is_anomaly_db,1],
            facecolors='none', edgecolors='green', s=120, label='DBSCAN Anomaly')

# cluster centers
plt.scatter(centers[:,0], centers[:,1], color='orange', marker='X', s=200, label='Cluster Centers')

plt.xlabel(feature_names[0])
plt.ylabel(feature_names[1])
plt.title("Clustering-Based Anomaly Detection (Iris 2D)")
plt.legend()
plt.show()

# ----------------------------
# 5) Bar plots for K-Means distances
# ----------------------------
plt.figure(figsize=(8,4))
plt.bar(range(len(dist_to_center)), dist_to_center, color='skyblue', edgecolor='k')
plt.axhline(threshold_km, color='red', linestyle='--', label='K-Means Distance Threshold')
plt.xlabel("Sample index")
plt.ylabel("Distance to Cluster Center")
plt.title("K-Means Anomaly Scores")
plt.legend()
plt.show()

# Tutorial: PCA-Based (Reconstructive) Anomaly Detection

Reconstructive methods detect anomalies by how **poorly a point can be reconstructed** from a lower-dimensional approximation of the data.  

## Steps:

1. Apply **Principal Component Analysis (PCA)** to reduce dimensionality:  
$$
X \approx \hat{X} = X_{\text{proj}} W^\top = X W W^\top
$$
where $W$ contains the top principal components.

2. Compute **reconstruction error** for each point:  
$$
\text{Error}_i = \| x_i - \hat{x}_i \|
$$

3. Points with the **largest reconstruction errors** are flagged as anomalies.

## Visualization:

- **Red points** → PCA reconstruction anomalies  
- **Orange X** → reconstructed points from the lower-dimensional subspace  

> PCA-based detection is useful when normal points lie near a **low-dimensional structure**, and anomalies deviate from it.

In [ ]:
# ----------------------------
# PCA-Based (Reconstructive) Anomaly Detection on Iris (2D)
# ----------------------------
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris
from sklearn.decomposition import PCA

# ----------------------------
# 1) Load Iris dataset (2D)
# ----------------------------
iris = load_iris()
X = iris.data[:, :2]  # first two features
feature_names = iris.feature_names[:2]

# ----------------------------
# 2) PCA Reconstruction
# ----------------------------
# Keep only 1 principal component for reconstruction
pca = PCA(n_components=1)
X_pca = pca.fit_transform(X)
X_recon = pca.inverse_transform(X_pca)

# ----------------------------
# 3) Compute reconstruction error
# ----------------------------
recon_error = np.linalg.norm(X - X_recon, axis=1)

# Flag top-n highest reconstruction errors as anomalies
num_anomalies = 5
threshold_recon = np.sort(recon_error)[-num_anomalies]
is_anomaly_recon = recon_error >= threshold_recon

# ----------------------------
# 4) Visualization
# ----------------------------
plt.figure(figsize=(7,7))

# Original points
plt.scatter(X[:,0], X[:,1], c='skyblue', s=50, label='Points')

# Anomalies
plt.scatter(X[is_anomaly_recon,0], X[is_anomaly_recon,1],
            color='red', s=80, label='PCA Reconstruction Anomaly', edgecolor='k')

# Reconstructed points (optional)
plt.scatter(X_recon[:,0], X_recon[:,1], color='orange', marker='x', s=80, label='Reconstructed Points')

plt.xlabel(feature_names[0])
plt.ylabel(feature_names[1])
plt.title("PCA-Based Reconstructive Anomaly Detection (Iris 2D)")
plt.legend()
plt.show()

# ----------------------------
# 5) Optional: Bar plot of reconstruction errors
# ----------------------------
plt.figure(figsize=(8,4))
plt.bar(range(len(recon_error)), recon_error, color='skyblue', edgecolor='k')
plt.axhline(threshold_recon, color='red', linestyle='--', label='Reconstruction Error Threshold')
plt.xlabel("Sample index")
plt.ylabel("Reconstruction Error")
plt.title("PCA-Based Anomaly Scores")
plt.legend()
plt.show()

# Exercise(s): Anomaly detection





## Exercise 1: Load and Explore Credit Card Fraud Dataset

Here we explore the famous Kaggle credit card fraud dataset:

https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud

We will downsample the dataset for faster calculations.

### Tasks:  
1. Compute and print:
   - Total number of transactions  
   - Number and percentage of **fraudulent transactions** (`Class = 1`)  
2. Visualize the **class distribution** with a bar chart.  
3. Plot histograms of key features such as `Amount` and `Time`.  
4. Generate a **scatter plot of transactions vs Time**, highlighting anomalies (`Class = 1`) in red.  
5. Apply PCA on the PCA-transformed features `V1–V28` and:
   - Reduce the data to 3D  
   - Plot a **3D PCA scatter plot**, coloring anomalies (`Class = 1`) in red  
   - Print the explained variance ratio for each component and the cumulative variance  
6. Write a short note on:
   - The **class imbalance** in the dataset  
   - Why this imbalance is important for anomaly detection methods
   - How PCA helps visualize rare events and what information might be lost

In [ ]:
# Write your code for the exercise here!

import pandas as pd

# URL for the credit card fraud dataset CSV
url = "https://raw.githubusercontent.com/nsethi31/Kaggle-Data-Credit-Card-Fraud-Detection/refs/heads/master/creditcard.csv"

# Load directly into a DataFrame
data = pd.read_csv(url)

# Inspect the full dataset
print("Original dataset shape:", data.shape)
print(data.head())
print("Original dataset class distribution:\n", data["Class"].value_counts())

# ----------------------------
# Downsample for lab use
# ----------------------------
# Separate majority and minority classes
df_fraud = data[data['Class'] == 1]
df_normal = data[data['Class'] == 0]

# Downsample normal transactions to 10x number of fraud cases
n_fraud = len(df_fraud)
df_normal_downsampled = df_normal.sample(n=n_fraud*10, random_state=42)

# Combine the downsampled normal with all fraud cases
data_downsampled = pd.concat([df_normal_downsampled, df_fraud]).sample(frac=1, random_state=42)  # shuffle

### Explain your findings here:




## Exercise 2: Statistical Anomaly Detection

In this exercise, we use **statistical methods** to detect anomalies in the credit card dataset.  
We will compute the **multivariate Gaussian model** on the PCA-transformed features `V1–V28` and flag rare points as anomalies.

### Tasks:

1. **Select features**:
   - Use the PCA-transformed features `V1–V28` only (exclude `Time` and `Amount`).  

2. **Compute statistics**:
   - Calculate the **mean vector** $\mu$ and **covariance matrix** $\Sigma$ of the selected features.
   - Recall the **multivariate Gaussian PDF**:
     $$
     p(\mathbf{x}) = \frac{1}{\sqrt{(2\pi)^d |\Sigma|}} \exp\Big(-\frac{1}{2} (\mathbf{x}-\mu)^T \Sigma^{-1} (\mathbf{x}-\mu)\Big)
     $$
     where $d$ = number of features.  

3. **Compute Mahalanobis distances** for each transaction:
   $$
   D_M(\mathbf{x}) = \sqrt{(\mathbf{x}-\mu)^T \Sigma^{-1} (\mathbf{x}-\mu)}
   $$

4. **Flag anomalies**:
   - Use a **chi-square threshold** (e.g., 95th percentile of $\chi^2_d$) or a top‑n heuristic.  
   - Compare flagged anomalies to the true fraud labels (`Class = 1`).

5. **Visualizations**:
   - Histogram or bar plot of **Mahalanobis distances**  
   - Scatter plot of **top 2 PCA components**, coloring anomalies in red, plot **p(x)** (probability density) over top PCA components as a contour  
   - Scatter plot of **top 3 PCA components**, coloring anomalies in red  

6. **Analysis**:
   - How many of the flagged anomalies are true fraud cases?  
   - How does the statistical method handle extreme but normal points?  
   - Discuss limitations of assuming a multivariate Gaussian for this dataset.

In [ ]:
# Write your code for the exercise here!

### Explain your findings here:




## Exercise 3: Distance- and Density-Based Anomaly Detection

In this exercise, we use **distance- and density-based methods** to identify anomalies (fraudulent transactions) in the credit card dataset.

We will use the **PCA-transformed features** (`V1–V28`) to compute distances between points and estimate densities.

### Tasks:

1. **Select features**:
   - Use PCA-transformed features `V1–V28` (exclude `Time` and `Amount`).

2. **Distance-based anomaly detection**:
   - Compute the **average distance to k nearest neighbors** for each transaction (exclude self-distance).  
   - Flag transactions with the **largest average distances** as anomalies.  

3. **Density-based anomaly detection**:
   - Compute a **relative density score**:
     $$
     \text{density}(x) = \frac{1}{\text{avg distance to k nearest neighbors}}
     $$
   - Transactions with **lowest densities** are anomalies.  

4. **Hyperparameters**:
   - Experiment with **k** (number of neighbors) and **number of anomalies to flag**.  
   - Discuss how these choices affect results.  

5. **Visualization**:
   - Scatter plot of top 3 PCA components, coloring **distance anomalies in red** and **density anomalies in green**.  
   - Bar plots of **distance scores** and **density scores** with thresholds marked.

6. **Analysis**:
   - Compare flagged anomalies to true fraud labels (`Class = 1`).  
   - Discuss strengths and limitations of distance- and density-based approaches for **rare events** like credit card fraud.  

In [ ]:
# Write your code for the exercise here!

### Explain your findings here:




## Exercise 4: Clustering-Based Anomaly Detection

In this exercise, we use **clustering methods** to detect anomalies in the credit card dataset.  
The idea is that transactions **far from cluster centers** or **marked as noise** may correspond to fraud.

### Tasks:

1. **Select features**:
   - Use PCA-transformed features `V1–V28` (exclude `Time` and `Amount`).

2. **K-Means Clustering**:
   - Fit **K-Means** with a chosen number of clusters (e.g., 3–5).  
   - Compute the **distance of each point to its cluster center**.  
   - Flag the **top-n farthest points** as anomalies.  

3. **DBSCAN Clustering**:
   - Fit **DBSCAN** with appropriate `eps` and `min_samples` for these features.  
   - Points labeled **-1** are considered anomalies (noise).  

4. **Visualization**:
   - Scatter plot of the **top 3 PCA components**:  
     - **K-Means anomalies in red**  
     - **DBSCAN anomalies in green**  
     - Cluster centers (for K-Means) in orange.  
   - Bar plot of **distance to cluster center** with threshold line.  

5. **Analysis**:
   - Compare flagged anomalies to true fraud labels (`Class = 1`).  
   - Discuss how anomalies **between clusters** or **on the edge of clusters** are treated.  
   - Reflect on strengths and weaknesses of **clustering approaches** for detecting rare events.

In [ ]:
# Write your code for the exercise here!

### Explain your findings here:




## Exercise 5: Reconstructive (PCA) Anomaly Detection

In this exercise, we use **reconstructive anomaly detection**.  
The idea is that **fraudulent transactions may not be well-represented by the principal components**, leading to **large reconstruction errors**.

### Tasks:

1. **Select features**:
   - Use PCA-transformed features `V1–V28` (exclude `Time` and `Amount`).

2. **Fit PCA**:
   - Fit **PCA with $n\_components < 28$** (e.g., enough to explain ~95% variance).  
   - Compute the **principal components** $ \mathbf{W} $ and the **mean vector** $ \mu $.

3. **Reconstruct data**:
   - Center the data: $ \mathbf{x}_c = \mathbf{x} - \mu $  
   - Project onto principal components: $ \mathbf{z} = \mathbf{x}_c \mathbf{W} $  
   - Reconstruct: $ \hat{\mathbf{x}} = \mathbf{z} \mathbf{W}^\top + \mu $

4. **Compute reconstruction error**:
   - Use **squared error**:  
     $$
     \text{error}(\mathbf{x}) = \|\mathbf{x} - \hat{\mathbf{x}}\|^2
     $$
   - Transactions with **highest reconstruction error** are flagged as anomalies.

5. **Visualization**:
   - Histogram or bar plot of **reconstruction errors**, marking threshold.  
   - Scatter plot of **top 3 PCA components**, coloring anomalies in red.

6. **Analysis**:
   - Compare flagged anomalies to true fraud labels (`Class = 1`).  
   - Discuss which types of anomalies **are caught by reconstructive methods** vs. distance-based or clustering methods.  
   - Reflect on limitations: e.g., if fraud transactions lie **along the same subspace** as normal ones, they may not produce large reconstruction error.

In [ ]:
# Write your code for the exercise here!

### Explain your findings here:




## Exercise 6: Modern Anomaly Detection with Scikit-Learn

In this exercise, we explore **state-of-the-art anomaly detection algorithms** using scikit-learn.  
These methods can automatically detect rare events such as fraud without manually computing distances or densities.

### Tasks:

1. **Select features**:
   - Use PCA-transformed features `V1–V28` (exclude `Time` and `Amount`).

2. **Try one or more scikit-learn anomaly detection methods**:
   - **Isolation Forest** (`sklearn.ensemble.IsolationForest`)  
   - **Local Outlier Factor (LOF)** (`sklearn.neighbors.LocalOutlierFactor`)  
   - **One-Class SVM** (`sklearn.svm.OneClassSVM`)  

3. **Fit the model**:
   - Train the algorithm on the dataset (you can use **all transactions** or only normal transactions for semi-supervised methods).  
   - Obtain **anomaly scores** or predicted labels.  

4. **Flag anomalies**:
   - Use the **top-n anomaly scores** or the model’s `predict` output.  
   - Compare flagged anomalies to the **true fraud labels** (`Class = 1`).  

5. **Visualization**:
   - Scatter plot of **top 3 PCA components**, coloring anomalies in red.  
   - Optional: histogram or bar plot of **anomaly scores**, marking threshold.  

6. **Analysis**:
   - Compare results to the **distance-based, density-based, clustering, and reconstructive methods**.  
   - Discuss which methods perform best for **rare, high-dimensional events**.  
   - Reflect on trade-offs: interpretability, computational cost, and sensitivity to hyperparameters.

In [ ]:
# Write your code for the exercise here!

### Explain your findings here:




# HTML Export Code

In [ ]:
# code to export notebook as .html for Canvas upload

from google.colab import drive
from google.colab import files

drive.mount('/content/drive')

notebook_name = "Lab_8"
!cp "/content/drive/MyDrive/Colab Notebooks/DSC 430/{notebook_name}.ipynb" /content/
!jupyter nbconvert --to html "/content/{notebook_name}.ipynb"
files.download(f"/content/{notebook_name}.html")

